# LoRA training on Colab (remote mode)

This notebook is the Colab half of `TRAIN_BACKEND=colab`. It does **not**
talk to the app; both sides trade files in one Google Drive folder:

```
LORA_TRAINING/jobs/<job_id>/
  job.json      <- the app writes the config and the dataset location
  dataset/      <- the app writes the images and their captions
  status.json   <- this notebook writes state, epoch, progress, error
  cancel.flag   <- the app writes this to ask for a clean stop
  output/       <- this notebook writes the .safetensors here
  manifest.json <- this notebook writes names, sizes and sha256
```

Nothing is faked. If training fails, `status.json` says `FAILED` with the
reason; no placeholder weights are written.

### How to run

1. Runtime -> Change runtime type -> **GPU**.
2. Set `DRIVE_ROOT` below to the Drive folder the app uses
   (`COLAB_DRIVE_ROOT` on Server-2).
3. Leave `JOB_ID` empty to pick the newest `QUEUED` job, or paste one id.
4. Run all. The notebook trains, honours `cancel.flag` between epochs, and
   can be re-run after a disconnect: it resumes and skips finished output.


## 1. Mount Google Drive and find the job


In [ ]:
import os, json, glob, time, hashlib, subprocess, sys

# The Drive folder shared with the app. Change this to your own path.
DRIVE_ROOT = os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive/LORA_TRAINING')
JOB_ID = os.environ.get('JOB_ID', '')  # empty = newest QUEUED job

from google.colab import drive
drive.mount('/content/drive')

def job_dir(job_id):
    return os.path.join(DRIVE_ROOT, 'jobs', job_id)

def read_json(path, default=None):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return default

def write_json(path, doc):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(doc, f, indent=2)
    os.replace(tmp, path)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def set_status(state, **extra):
    doc = {'state': state, 'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
    doc.update(extra)
    write_json(os.path.join(job_dir(job_id), 'status.json'), doc)
    print('status ->', doc)

def pick_job():
    if JOB_ID:
        return JOB_ID
    jobs = sorted(glob.glob(os.path.join(DRIVE_ROOT, 'jobs', '*')))
    for d in reversed(jobs):
        st = read_json(os.path.join(d, 'status.json'), {})
        if st.get('state', 'QUEUED') == 'QUEUED':
            return os.path.basename(d)
    return os.path.basename(jobs[-1]) if jobs else ''

job_id = pick_job()
assert job_id, f'Walang job sa {DRIVE_ROOT}/jobs'
print('job:', job_id)
print('job dir:', job_dir(job_id))


## 2. Read job.json (the single source of truth)

Every training parameter comes from `job.json`, which the app wrote from the
same kohya config the local trainer would run. Nothing is typed by hand here,
so a Colab run and a local run train the same way.


In [ ]:
job = read_json(os.path.join(job_dir(job_id), 'job.json'))
assert job, 'Hindi mabasa ang job.json'
args = job['train_args']
print(json.dumps(job, indent=2))

# Resolve the paths relative to DRIVE_ROOT.
dataset_dir = os.path.join(DRIVE_ROOT, job['dataset'])
output_dir  = os.path.join(DRIVE_ROOT, job['output'])
os.makedirs(output_dir, exist_ok=True)
print('dataset:', dataset_dir)
print('output :', output_dir)
print('samples:', len(glob.glob(os.path.join(dataset_dir, '*.png'))))


## 3. Install sd-scripts (kohya) and fetch the base model

The tools come from GitHub (sd-scripts) and HuggingFace (the base model).
Tokens are read from **Colab Secrets** (left sidebar -> key icon), never
written here:

| Secret | Used for |
| --- | --- |
| `HF_TOKEN` | a gated / private HuggingFace base model |
| `GITHUB_TOKEN` | a private sd-scripts fork, or higher clone rate limits |
| `SD_SCRIPTS_REPO` | optional: override the sd-scripts repo URL |

A missing secret is not fatal: the notebook falls back to the public,
anonymous download and says which token it did or did not find.


In [ ]:
# --- Tool fetching: tokens from Colab Secrets (userdata.get) ----------------
# Nothing secret is written in this notebook. Add the keys in the left sidebar
# (the key icon), then enable "Notebook access" for each:
#   HF_TOKEN         HuggingFace token, for a gated/private base model
#   GITHUB_TOKEN     GitHub token, for a private sd-scripts fork / rate limits
#   SD_SCRIPTS_REPO  optional override of the sd-scripts repo URL
from google.colab import userdata

def secret(name, default=''):
    try:
        return userdata.get(name) or default
    except Exception:
        return default  # the secret is simply not set

HF_TOKEN = secret('HF_TOKEN')
GITHUB_TOKEN = secret('GITHUB_TOKEN')
SD_SCRIPTS_REPO = secret('SD_SCRIPTS_REPO', 'https://github.com/kohya-ss/sd-scripts.git')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN  # huggingface_hub reads this for gated models
    print('HF_TOKEN: nakita (kaya ang gated/private base model)')
else:
    print('HF_TOKEN: wala — public na base model lang ang ma-download.')

# --- sd-scripts is the trainer the local mode uses (KOHYA_DIR) -------------
# Both modes train with the same code. A GITHUB_TOKEN is only needed for a
# private fork; for the public repo it just raises the rate limit.
clone_url = SD_SCRIPTS_REPO
if GITHUB_TOKEN and 'github.com' in clone_url and '@' not in clone_url:
    clone_url = clone_url.replace('https://', 'https://' + GITHUB_TOKEN + '@')
print('sd-scripts:', SD_SCRIPTS_REPO)
!git clone -q "$clone_url" /content/sd-scripts || git -C /content/sd-scripts pull -q
# Do not leave the token in .git/config on the VM.
!git -C /content/sd-scripts remote set-url origin "$SD_SCRIPTS_REPO"
%cd /content/sd-scripts
!pip install -q -r requirements.txt
!pip install -q bitsandbytes
%cd /content

# --- The base model: a HuggingFace id or a path inside Drive ---------------
# job.json records it, so this is not a guess.
base_model = job['base_model']
if '/' in base_model and not os.path.exists(base_model):
    from huggingface_hub import snapshot_download
    local_base = snapshot_download(
        base_model,
        local_dir=job.get('colab_base_model_dir', '/content/models'),
        token=HF_TOKEN or None,
    )
else:
    local_base = base_model
print('base model:', local_base)


## 4. Write the kohya config from job.json, then train

The config is generated from `job['train_args']`, not typed by hand. The
training loop checks `cancel.flag` between epochs and exits cleanly.


In [ ]:
config_path = os.path.join(job_dir(job_id), 'kohya_config.toml')
config = f'''# Generated on Colab from job.json
[general]
enable_bucket = true
bucket_no_upscale = false
output_dir = "{output_dir}"
output_name = "{job['output_name']}"
save_model_as = "{args['save_model_as']}"

[[datasets]]
resolution = {args['resolution']}
batch_size = {args['batch_size']}
enable_bucket = true

  [[datasets.subsets]]
  image_dir = "{dataset_dir}"
  caption_extension = ".txt"
  num_repeats = {args['num_repeats']}

[network_arguments]
network_module = "{args['network_module']}"
network_dim = {args['network_dim']}
network_alpha = {args['network_alpha']}

[optimizer_arguments]
learning_rate = {args['learning_rate']}
optimizer_type = "{args['optimizer_type']}"
lr_scheduler = "{args['lr_scheduler']}"

[training_arguments]
pretrained_model_name_or_path = "{local_base}"
max_train_epochs = {args['max_train_epochs']}
save_every_n_epochs = {args['save_every_n_epochs']}
mixed_precision = "{args['mixed_precision']}"
seed = {args['seed']}

[logging_arguments]
logging_dir = "{output_dir}/logs"
'''
with open(config_path, 'w') as f:
    f.write(config)
print(config)


In [ ]:
cancel_flag = os.path.join(job_dir(job_id), 'cancel.flag')

# Resume: skip a run whose output already exists (a rerun after a disconnect
# keeps what finished). A fresh job has an empty output/ folder.
existing = glob.glob(os.path.join(output_dir, '*.safetensors'))
if existing and not os.path.exists(cancel_flag):
    print('May umiiral nang output, laktawan ang training:', existing)
else:
    set_status('RUNNING', epoch=0, total_epochs=args['max_train_epochs'])
    cmd = [sys.executable, '/content/sd-scripts/sdxl_train_network.py',
           '--config_file', config_path]
    print('$', ' '.join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    epoch = 0
    for line in proc.stdout:
        print(line, end='')
        if 'epoch' in line.lower():
            epoch += 1
            set_status('RUNNING', epoch=epoch, total_epochs=args['max_train_epochs'],
                       message=line.strip()[:200])
        if os.path.exists(cancel_flag):
            proc.terminate()
            proc.wait(timeout=30)
            set_status('CANCELLED', epoch=epoch, message='cancel.flag nakita')
            raise SystemExit('Cancelled sa pagitan ng epoch.')
    code = proc.wait()
    if code != 0:
        set_status('FAILED', error=f'trainer exited with code {code}')
        raise SystemExit(f'FAILED (code {code})')
    print('training done')


## 5. Write manifest.json (names, sizes, sha256)


In [ ]:
weights = sorted(glob.glob(os.path.join(output_dir, '*.safetensors')))
assert weights, f'Walang .safetensors sa {output_dir}'
files = [{'name': os.path.basename(w),
          'bytes': os.path.getsize(w),
          'sha256': sha256_file(w)} for w in weights]
write_json(os.path.join(job_dir(job_id), 'manifest.json'), {'files': files})
print(json.dumps(files, indent=2))
set_status('DONE', message=f'{len(files)} file(s)')
print('Tapos. Hihintayin ng app ang DONE at i-verify ang sha256.')
